# CNN and Convolution Mathematics — Theory Notebook

Interactive explorations of:
- The convolution operation and its linear-algebra representation
- Backpropagation through convolutional layers
- Batch Normalisation (forward & backward)
- Depthwise-separable, dilated, and transposed convolutions
- Residual connections and skip-gradient analysis
- EfficientNet compound scaling
- ViT patch embedding equivalence

Each section pairs the mathematics from `notes.md` with executable code and visualisations.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle, FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
print("Libraries loaded ✓")


## 1  Convolution vs Cross-Correlation

**True convolution** (signal processing):
$$
(f * g)[n] = \sum_{k} f[k]\, g[n - k]
$$

**Cross-correlation** (what deep learning frameworks implement):
$$
(f \star g)[n] = \sum_{k} f[k]\, g[n + k]
$$

The difference is a kernel flip.  For *learned* kernels the distinction is irrelevant
(the network can learn the flipped kernel), but it matters when using known filters
(Sobel, Laplacian, etc.).


In [ ]:
def true_conv_1d(signal, kernel):
    """True 1-D convolution: NumPy handles the kernel flip internally."""
    return np.convolve(signal, kernel, mode='same')

def cross_corr_1d(signal, kernel):
    """Cross-correlation: slide kernel without flipping."""
    return np.correlate(signal, kernel, mode='same')

# Asymmetric kernel so the difference is visible
signal = np.zeros(30); signal[10:20] = 1.0        # rectangular pulse
kernel = np.array([0.5, 1.0, 4.0, 2.0, 1.0])     # asymmetric ramp-right

conv_out  = true_conv_1d(signal, kernel)
xcorr_out = cross_corr_1d(signal, kernel)

fig, axes = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
axes[0].plot(signal, 'k', lw=2); axes[0].set_ylabel('Signal'); axes[0].set_title('Input Signal')
axes[1].stem(np.arange(len(kernel)) - len(kernel)//2, kernel,
             basefmt='k-', linefmt='C1-', markerfmt='C1o')
axes[1].set_ylabel('Kernel'); axes[1].set_title('Filter Kernel (asymmetric)')
axes[2].plot(conv_out, 'C0', lw=2); axes[2].set_ylabel('True conv')
axes[2].set_title('True Convolution (kernel flipped internally)')
axes[3].plot(xcorr_out, 'C2', lw=2); axes[3].set_ylabel('Cross-corr')
axes[3].set_title('Cross-Correlation (PyTorch default)')
plt.tight_layout()
plt.suptitle('Convolution vs Cross-Correlation', y=1.01, fontsize=13, fontweight='bold')
plt.show()

print('Are they the same?', np.allclose(conv_out, xcorr_out))
print('They differ when the kernel is asymmetric.')


## 2  2-D Convolution Step-by-Step

A 2-D cross-correlation with a $3\times3$ kernel:

$$
(\mathbf{X} \star \mathbf{K})[i,j] = \sum_{m=0}^{k-1}\sum_{n=0}^{k-1}
    \mathbf{X}[i+m,\, j+n]\cdot \mathbf{K}[m,n]
$$

Output size (no padding, stride 1):
$$
H_{\text{out}} = H_{\text{in}} - k + 1, \quad W_{\text{out}} = W_{\text{in}} - k + 1
$$


In [ ]:
def conv2d_naive(X, K, stride=1, padding=0):
    """Pure-NumPy 2-D cross-correlation."""
    if padding:
        X = np.pad(X, padding)
    H, W   = X.shape
    kH, kW = K.shape
    oH = (H - kH) // stride + 1
    oW = (W - kW) // stride + 1
    out = np.zeros((oH, oW))
    for i in range(oH):
        for j in range(oW):
            patch = X[i*stride:i*stride+kH, j*stride:j*stride+kW]
            out[i, j] = np.sum(patch * K)
    return out

# Horizontal edge detector
X = np.array([
    [0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0],
    [0,1,1,1,1,1,0],
    [0,1,1,1,1,1,0],
    [0,1,1,1,1,1,0],
    [0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0],
], dtype=float)

K_horiz = np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=float)
K_vert  = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=float)
K_lap   = np.array([[0,-1,0],[-1,4,-1],[0,-1,0]], dtype=float)

out_horiz = conv2d_naive(X, K_horiz, padding=1)
out_vert  = conv2d_naive(X, K_vert,  padding=1)
out_lap   = conv2d_naive(X, K_lap,   padding=1)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
titles = ['Input', 'Horiz. Kernel', 'Vert. Kernel', 'Laplacian Kernel',
          'Input', 'Horiz. Edges', 'Vert. Edges', 'Laplacian']
imgs   = [X, K_horiz, K_vert, K_lap, X, out_horiz, out_vert, out_lap]
cmaps  = ['gray','RdBu','RdBu','RdBu','gray','RdBu','RdBu','RdBu']
for ax, img, title, cmap in zip(axes.flat, imgs, titles, cmaps):
    im = ax.imshow(img, cmap=cmap, aspect='auto')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('2-D Convolution with Classical Filters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3  Linear-Algebra View: Toeplitz Matrices

A 1-D convolution with kernel $\mathbf{k}=[k_0,k_1,k_2]$ acting on a length-5 signal
is equivalent to multiplication by a **Toeplitz matrix**:

$$
\begin{pmatrix}k_0 & 0 & 0 & 0 & 0 \\ k_1 & k_0 & 0 & 0 & 0 \\
k_2 & k_1 & k_0 & 0 & 0 \\ 0 & k_2 & k_1 & k_0 & 0 \\ 0 & 0 & k_2 & k_1 & k_0
\end{pmatrix}\mathbf{x}
$$

Key consequence: the **weight matrix is not full-rank** — massive parameter savings vs FC.


In [ ]:
def conv_toeplitz_1d(x, k):
    """1-D same-convolution via an explicit Toeplitz linear operator."""
    n = len(x)
    basis = np.eye(n)
    T = np.column_stack([np.convolve(basis[:, j], k, mode='same') for j in range(n)])
    return T @ x, T

x = np.array([1., 2., 3., 4., 5.])
k = np.array([0.5, 1.0, 0.5])

y_direct  = np.convolve(x, k, mode='same')
y_toep, T = conv_toeplitz_1d(x, k)

print('Direct  conv:', np.round(y_direct, 4))
print('Toeplitz conv:', np.round(y_toep, 4))
print('Match:', np.allclose(y_direct, y_toep))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im = axes[0].imshow(T, cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_title('Toeplitz Matrix T'); plt.colorbar(im, ax=axes[0])
axes[0].set_xlabel('Input index'); axes[0].set_ylabel('Output index')

axes[1].bar(range(len(x)), x, color='C0', alpha=0.7, label='Input x')
axes[1].bar(range(len(y_toep)), y_toep, color='C1', alpha=0.7, label='Output Tx')
axes[1].legend(); axes[1].set_title('Input vs Convolved Output')
plt.tight_layout()
plt.show()

rank = np.linalg.matrix_rank(T)
print(f'\nToeplitz matrix rank = {rank} (size {T.shape})')
print(f'A full-rank square matrix would need {T.shape[0]**2} parameters; '
      f'this conv uses only {len(k)} parameters.')


## 4  Convolution Theorem: FFT Acceleration

$$
\mathcal{F}\{f * g\} = \mathcal{F}\{f\} \cdot \mathcal{F}\{g\}
$$

Direct convolution: $O(N \cdot k^2)$ multiplications.
FFT-based convolution: $O(N \log N)$ — huge win for large kernels.


In [ ]:
import time
from numpy.fft import fft2, ifft2

def conv_fft_2d(X, K, padding=0):
    """2-D cross-correlation via FFT with zero-padding for linear convolution."""
    if padding:
        X = np.pad(X, padding)
    H, W = X.shape
    kH, kW = K.shape
    shape = (H + kH - 1, W + kW - 1)
    K_flip = np.flip(K, axis=(0, 1))
    full = np.real(ifft2(fft2(X, shape) * fft2(K_flip, shape)))
    start_h, start_w = kH - 1, kW - 1
    end_h = start_h + H - kH + 1
    end_w = start_w + W - kW + 1
    return full[start_h:end_h, start_w:end_w]

# Benchmark: direct vs FFT for increasing image size
sizes = [32, 64, 128, 256]
k_size = 15
direct_times, fft_times = [], []
pad = k_size // 2

for sz in sizes:
    img = np.random.randn(sz, sz)
    ker = np.random.randn(k_size, k_size)

    t0 = time.perf_counter()
    for _ in range(5):
        conv2d_naive(img, ker, padding=pad)
    direct_times.append((time.perf_counter() - t0) / 5)

    t0 = time.perf_counter()
    for _ in range(5):
        conv_fft_2d(img, ker, padding=pad)
    fft_times.append((time.perf_counter() - t0) / 5)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, np.array(direct_times) * 1000, 'C0o-', lw=2, label='Direct O(Nk²)')
ax.plot(sizes, np.array(fft_times) * 1000, 'C1s-', lw=2, label='FFT O(N log N)')
ax.set_xlabel('Image size (N×N)'); ax.set_ylabel('Time per call (ms)')
ax.set_title(f'Direct vs FFT Convolution  (kernel {k_size}×{k_size})')
ax.legend(); ax.set_yscale('log')
plt.tight_layout(); plt.show()

# Verify correctness on small input
X_s = np.random.randn(32, 32)
K_s = np.random.randn(3, 3)
direct = conv2d_naive(X_s, K_s, padding=1)
fft_r = conv_fft_2d(X_s, K_s, padding=1)
print('Direct vs FFT results close?', np.allclose(direct, fft_r, atol=1e-10))


## 5  Backpropagation Through a Convolutional Layer

Given upstream gradient $\frac{\partial \mathcal{L}}{\partial \mathbf{Y}}$:

**Gradient w.r.t. input** (needed to propagate to previous layer):
$$
\frac{\partial \mathcal{L}}{\partial \mathbf{X}} = \frac{\partial \mathcal{L}}{\partial \mathbf{Y}} \star' \mathbf{K}^{\text{flip}}
$$
This is a *transposed convolution* (full convolution with flipped kernel).

**Gradient w.r.t. kernel** (needed to update weights):
$$
\frac{\partial \mathcal{L}}{\partial \mathbf{K}} = \mathbf{X} \star \frac{\partial \mathcal{L}}{\partial \mathbf{Y}}
$$


In [ ]:
def conv2d_forward(X, K):
    """Forward pass of 2-D cross-correlation (VALID padding)."""
    H, W   = X.shape
    kH, kW = K.shape
    oH, oW = H - kH + 1, W - kW + 1
    Y = np.zeros((oH, oW))
    for i in range(oH):
        for j in range(oW):
            Y[i, j] = np.sum(X[i:i+kH, j:j+kW] * K)
    return Y

def conv2d_backward(X, K, dY):
    """
    Backward pass.
    Returns dX (same shape as X) and dK (same shape as K).
    """
    H, W   = X.shape
    kH, kW = K.shape
    oH, oW = dY.shape

    # --- Gradient w.r.t. X: full convolution with flipped kernel ---
    dX = np.zeros_like(X)
    K_flip = K[::-1, ::-1]
    dY_padded = np.pad(dY, ((kH-1, kH-1), (kW-1, kW-1)))
    for i in range(H):
        for j in range(W):
            dX[i, j] = np.sum(dY_padded[i:i+kH, j:j+kW] * K_flip)

    # --- Gradient w.r.t. K: cross-correlation of X with dY ---
    dK = np.zeros_like(K)
    for m in range(kH):
        for n in range(kW):
            dK[m, n] = np.sum(X[m:m+oH, n:n+oW] * dY)

    return dX, dK

# ── Numerical gradient check ────────────────────────────────────────────────
X = np.random.randn(6, 6)
K = np.random.randn(3, 3)
eps = 1e-5

Y    = conv2d_forward(X, K)
dY   = np.ones_like(Y)                     # gradient of sum w.r.t. Y
dX, dK = conv2d_backward(X, K, dY)

# Numerical dX
dX_num = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        X[i,j] += eps; Y_p = conv2d_forward(X, K)
        X[i,j] -= 2*eps; Y_m = conv2d_forward(X, K)
        X[i,j] += eps
        dX_num[i,j] = (Y_p.sum() - Y_m.sum()) / (2*eps)

# Numerical dK
dK_num = np.zeros_like(K)
for i in range(K.shape[0]):
    for j in range(K.shape[1]):
        K[i,j] += eps; Y_p = conv2d_forward(X, K)
        K[i,j] -= 2*eps; Y_m = conv2d_forward(X, K)
        K[i,j] += eps
        dK_num[i,j] = (Y_p.sum() - Y_m.sum()) / (2*eps)

print("dX gradient check:", "PASS" if np.allclose(dX, dX_num, atol=1e-6) else "FAIL")
print("dK gradient check:", "PASS" if np.allclose(dK, dK_num, atol=1e-6) else "FAIL")
print(f"  max |dX_analytic - dX_numeric| = {np.max(np.abs(dX - dX_num)):.2e}")
print(f"  max |dK_analytic - dK_numeric| = {np.max(np.abs(dK - dK_num)):.2e}")


## 6  im2col: Convolution as Matrix Multiplication

The `im2col` trick converts a conv into a single GEMM (General Matrix-Matrix Multiply),
which is heavily optimised on GPUs via cuBLAS.

For input $\mathbf{X}\in\mathbb{R}^{C_{\text{in}}\times H\times W}$ and kernel size $k$:

1. Extract every $k\times k$ patch → matrix $\mathbf{P}\in\mathbb{R}^{(C_{\text{in}}k^2)\times(H'W')}$
2. Reshape kernel → matrix $\mathbf{W}\in\mathbb{R}^{C_{\text{out}}\times(C_{\text{in}}k^2)}$
3. Output $\mathbf{Y} = \mathbf{W}\mathbf{P}\in\mathbb{R}^{C_{\text{out}}\times(H'W')}$


In [ ]:
def im2col(X, kH, kW, stride=1, padding=0):
    """
    X: (C, H, W)
    Returns P: (C*kH*kW, H_out*W_out)
    """
    C, H, W = X.shape
    if padding:
        X = np.pad(X, ((0,0),(padding,padding),(padding,padding)))
        _, H, W = X.shape
    H_out = (H - kH) // stride + 1
    W_out = (W - kW) // stride + 1
    cols = []
    for i in range(0, H - kH + 1, stride):
        for j in range(0, W - kW + 1, stride):
            patch = X[:, i:i+kH, j:j+kW].ravel()  # (C*kH*kW,)
            cols.append(patch)
    return np.stack(cols, axis=1)  # (C*kH*kW, H_out*W_out)

# Example: 3-channel 8×8 input, 3×3 kernel, 16 output channels
C_in, H, W = 3, 8, 8
C_out, kH, kW = 16, 3, 3

X_mc = np.random.randn(C_in, H, W)
W_mc = np.random.randn(C_out, C_in * kH * kW)  # kernel matrix

P  = im2col(X_mc, kH, kW, padding=1)            # (C_in*kH*kW, H*W)
Y_gemm = W_mc @ P                               # (C_out, H*W)

print(f"Input shape:   {X_mc.shape}")
print(f"im2col shape:  {P.shape}  (C_in*kH*kW={C_in*kH*kW}, H*W={H*W})")
print(f"Kernel matrix: {W_mc.shape}")
print(f"Output (GEMM): {Y_gemm.shape}")

# Compare with naive conv (single channel for simplicity)
X_1ch = X_mc[0]
K_1ch = W_mc[0, :kH*kW].reshape(kH, kW) / (C_in * kH * kW)
naive_out = conv2d_naive(X_1ch, K_1ch, padding=1).ravel()

# Memory usage analysis
input_mem   = X_mc.nbytes
col_mem     = P.nbytes
overhead    = col_mem / input_mem
print(f"\nim2col memory overhead: {overhead:.1f}x input size")
print(f"  Input: {input_mem} bytes, im2col: {col_mem} bytes")
print("  Trade-off: memory for speed (GEMM >> loop conv on GPU)")


## 7  Batch Normalisation: Forward and Backward Pass

**Forward pass** (training):
$$
\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad
y_i = \gamma\hat{x}_i + \beta
$$

**Backward pass** (from [Ioffe & Szegedy 2015], full derivation):
$$
\frac{\partial\mathcal{L}}{\partial x_i} = \frac{\gamma}{N\hat{\sigma}}
\left[N\frac{\partial\mathcal{L}}{\partial y_i} - \sum_j\frac{\partial\mathcal{L}}{\partial y_j}
- \hat{x}_i\sum_j\frac{\partial\mathcal{L}}{\partial y_j}\hat{x}_j\right]
$$


In [ ]:
class BatchNorm1D:
    """Full BatchNorm with analytical backward pass."""

    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        self.gamma = np.ones(num_features)
        self.beta  = np.zeros(num_features)
        self.eps   = eps
        self.momentum = momentum
        self.running_mean = np.zeros(num_features)
        self.running_var  = np.ones(num_features)
        # cache for backward
        self._cache = {}

    def forward(self, x, training=True):
        """x: (N, D)"""
        N, D = x.shape
        if training:
            mu    = x.mean(axis=0)           # (D,)
            var   = x.var(axis=0)             # (D,)
            x_hat = (x - mu) / np.sqrt(var + self.eps)
            out   = self.gamma * x_hat + self.beta
            self._cache = {'x': x, 'mu': mu, 'var': var, 'x_hat': x_hat, 'N': N}
            self.running_mean = (1 - self.momentum)*self.running_mean + self.momentum*mu
            self.running_var  = (1 - self.momentum)*self.running_var  + self.momentum*var
        else:
            x_hat = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)
            out   = self.gamma * x_hat + self.beta
        return out

    def backward(self, dout):
        """Returns dx, dgamma, dbeta."""
        x, mu, var, x_hat, N = (self._cache[k]
                                 for k in ['x','mu','var','x_hat','N'])
        sigma = np.sqrt(var + self.eps)

        dbeta  = dout.sum(axis=0)
        dgamma = (dout * x_hat).sum(axis=0)

        dx_hat = dout * self.gamma
        # Full analytical formula
        dx = (1/N) * (1/sigma) * (
            N * dx_hat
            - dx_hat.sum(axis=0)
            - x_hat * (dx_hat * x_hat).sum(axis=0)
        )
        return dx, dgamma, dbeta


# ── Numerical gradient check ────────────────────────────────────────────────
bn  = BatchNorm1D(4)
x_t = np.random.randn(8, 4)
out = bn.forward(x_t, training=True)
dout = np.random.randn(*out.shape)
dx, dgamma, dbeta = bn.backward(dout)

eps = 1e-5
dx_num = np.zeros_like(x_t)
for i in range(x_t.shape[0]):
    for j in range(x_t.shape[1]):
        x_t[i,j] += eps; o_p = bn.forward(x_t, training=True); bn._cache['x']=x_t.copy()
        x_t[i,j] -= 2*eps; o_m = bn.forward(x_t, training=True)
        x_t[i,j] += eps
        dx_num[i,j] = np.sum(dout * (o_p - o_m)) / (2*eps)

print("BatchNorm dx gradient check:",
      "PASS" if np.allclose(dx, dx_num, atol=1e-5) else "FAIL")
print(f"  max |error| = {np.max(np.abs(dx - dx_num)):.2e}")

# Visualise normalisation effect
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x_raw = np.random.randn(100, 4) * np.array([10, 0.1, 5, 2]) + np.array([5, -3, 0, 8])
x_norm = bn.forward(x_raw, training=True)
for i in range(4):
    axes[0].hist(x_raw[:, i], bins=20, alpha=0.6, label=f'feat {i}')
    axes[1].hist(x_norm[:, i], bins=20, alpha=0.6, label=f'feat {i}')
axes[0].set_title('Before BatchNorm'); axes[1].set_title('After BatchNorm')
for ax in axes: ax.legend(fontsize=8)
plt.suptitle('BatchNorm Normalisation Effect', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 8  Depthwise Separable Convolution

Standard conv: $C_{\text{in}}\cdot C_{\text{out}}\cdot k^2$ parameters.

Depthwise separable = depthwise conv + pointwise conv:
- Depthwise: $C_{\text{in}}\cdot k^2$ parameters
- Pointwise ($1\times1$): $C_{\text{in}}\cdot C_{\text{out}}$ parameters

**Reduction ratio:**
$$
\frac{C_{\text{in}} k^2 + C_{\text{in}} C_{\text{out}}}{C_{\text{in}} C_{\text{out}} k^2}
= \frac{1}{C_{\text{out}}} + \frac{1}{k^2}
$$


In [ ]:
def depthwise_conv(X, kernels_dw, stride=1, padding=1):
    """X: (C,H,W), kernels_dw: (C,kH,kW). Apply one kernel per channel."""
    C, H, W = X.shape
    kH, kW = kernels_dw.shape[1:]
    X_pad = np.pad(X, ((0,0),(padding,padding),(padding,padding)))
    oH = (H + 2*padding - kH) // stride + 1
    oW = (W + 2*padding - kW) // stride + 1
    out = np.zeros((C, oH, oW))
    for c in range(C):
        out[c] = conv2d_naive(X_pad[c], kernels_dw[c])
    return out

def pointwise_conv(X, W_pw):
    """X: (C_in,H,W), W_pw: (C_out,C_in). Returns (C_out,H,W)."""
    C_in, H, W = X.shape
    return np.einsum('ij,jhw->ihw', W_pw, X)

# Parameter count comparison
def param_count_standard(C_in, C_out, k):
    return C_in * C_out * k * k

def param_count_dw_sep(C_in, C_out, k):
    return C_in * k * k + C_in * C_out

configs = [(32, 64, 3), (64, 128, 3), (128, 256, 3), (256, 512, 3)]
print("C_in  C_out  k  Standard  DW-Sep  Reduction")
print("-" * 55)
for C_in, C_out, k in configs:
    std = param_count_standard(C_in, C_out, k)
    dws = param_count_dw_sep(C_in, C_out, k)
    red = std / dws
    print(f"{C_in:4d}  {C_out:5d}  {k}  {std:8d}  {dws:6d}  {red:.1f}x")

# Visualise
fig, ax = plt.subplots(figsize=(9, 4))
C_outs = np.arange(16, 513, 16)
k = 3; C_in = 64
std_params = [param_count_standard(C_in, c, k) for c in C_outs]
dws_params = [param_count_dw_sep(C_in, c, k) for c in C_outs]
ax.plot(C_outs, std_params, 'C0-', lw=2, label='Standard Conv')
ax.plot(C_outs, dws_params, 'C1--', lw=2, label='Depthwise-Separable')
ax.fill_between(C_outs, dws_params, std_params, alpha=0.15, color='C2', label='Saved params')
ax.set_xlabel('C_out'); ax.set_ylabel('Parameters')
ax.set_title(f'Parameter Count: Standard vs DW-Sep (C_in={C_in}, k={k})')
ax.legend(); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.tight_layout(); plt.show()


## 9  Dilated (Atrous) Convolution

With dilation rate $d$, a $k\times k$ kernel has **effective receptive field**:
$$
k_{\text{eff}} = k + (k-1)(d-1) = d(k-1)+1
$$

Stacking dilated conv with rates $1,2,4,\ldots,2^{L-1}$ gives receptive field
$\approx 2^L(k-1)+1$ with only $L\cdot k^2$ parameters.  Used in WaveNet.


In [ ]:
def dilated_conv2d(X, K, dilation=1):
    """2-D convolution with dilation (valid padding)."""
    H, W   = X.shape
    kH, kW = K.shape
    kH_eff = kH + (kH-1)*(dilation-1)
    kW_eff = kW + (kW-1)*(dilation-1)
    oH = H - kH_eff + 1
    oW = W - kW_eff + 1
    if oH <= 0 or oW <= 0:
        return np.array([[]])
    out = np.zeros((oH, oW))
    for i in range(oH):
        for j in range(oW):
            val = 0.
            for m in range(kH):
                for n in range(kW):
                    val += X[i + m*dilation, j + n*dilation] * K[m, n]
            out[i, j] = val
    return out

# Visualise receptive field growth with stacked dilations
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
size = 24
dilations = [1, 2, 4, 8]
titles = ['Dilation=1', 'Dilation=2', 'Dilation=4', 'Dilation=8']

for ax, d, title in zip(axes, dilations, titles):
    canvas = np.zeros((size, size))
    # Mark kernel positions for 3×3 kernel centred at (size//2, size//2)
    cx, cy = size//2, size//2
    kH, kW = 3, 3
    for m in range(kH):
        for n in range(kW):
            r = cy + (m - kH//2)*d
            c = cx + (n - kW//2)*d
            if 0 <= r < size and 0 <= c < size:
                canvas[r, c] = 1.0
    # Mark effective receptive field boundary
    keff = kH + (kH-1)*(d-1)
    r0 = cy - keff//2; c0 = cx - keff//2
    ax.imshow(canvas, cmap='Blues', vmin=0, vmax=1)
    rect = Rectangle((c0-.5, r0-.5), keff, keff,
                      linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.set_title(f'{title}\n(eff. RF = {keff}×{keff})')
    ax.axis('off')

plt.suptitle('Dilated Convolution: Kernel Positions and Effective Receptive Field',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Effective receptive field formula
print('Effective receptive field (3×3 kernel):')
for d in [1,2,4,8,16]:
    keff = 3 + (3-1)*(d-1)
    print(f'  dilation={d:3d} → effective RF = {keff}×{keff}')


## 10  Transposed Convolution (Deconvolution)

The transposed convolution is the *adjoint* of a convolution.  If convolution
down-samples by stride $s$, the transposed conv up-samples by $s$.

**Checkerboard condition** (avoid artifacts): choose kernel size $k$ such that
$k \bmod s = 0$.  Otherwise fractional coverage causes uneven outputs.


In [ ]:
def transposed_conv2d_1d(x, k, stride=2):
    """
    1-D transposed convolution (up-sampling).
    Inserts (stride-1) zeros between input elements, then applies full conv.
    """
    # Step 1: insert zeros (stride-1 between each element)
    x_expanded = np.zeros(len(x)*stride - (stride-1))
    x_expanded[::stride] = x

    # Step 2: full convolution (pad with k-1 zeros on each side)
    k_flip = k[::-1]
    x_padded = np.pad(x_expanded, len(k)-1)
    out_len = len(x_expanded) + len(k) - 1
    out = np.zeros(out_len)
    for i in range(out_len):
        patch = x_padded[i:i+len(k)]
        if len(patch) == len(k):
            out[i] = np.dot(patch, k_flip)
    return out

# Compare strides and kernel sizes for checkerboard analysis
x_1d = np.array([1., 2., 3., 2., 1.])
k_good = np.array([0.25, 0.5, 0.25])    # k=3, not divisible by 2 (slightly uneven)
k_best = np.array([0.25, 0.5, 0.5, 0.25])  # k=4, divisible by 2

out_k3 = transposed_conv2d_1d(x_1d, k_good, stride=2)
out_k4 = transposed_conv2d_1d(x_1d, k_best, stride=2)

fig, axes = plt.subplots(3, 1, figsize=(10, 7))
axes[0].bar(range(len(x_1d)), x_1d, color='C0', alpha=0.8, label='Input')
axes[0].set_title('Input (5 values)'); axes[0].set_ylabel('Value')

axes[1].bar(range(len(out_k3)), out_k3, color='C1', alpha=0.8)
axes[1].set_title('Transposed Conv (stride=2, k=3, k mod s = 1: checkerboard risk)')
axes[1].set_ylabel('Value')

axes[2].bar(range(len(out_k4)), out_k4, color='C2', alpha=0.8)
axes[2].set_title('Transposed Conv (stride=2, k=4, k mod s = 0: uniform coverage)')
axes[2].set_ylabel('Value')

for ax in axes: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('Transposed Convolution: Checkerboard Artifact Analysis',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

var_k3 = np.var(out_k3)
var_k4 = np.var(out_k4)
print(f"Output variance — k=3 (bad): {var_k3:.4f},  k=4 (good): {var_k4:.4f}")
print("Lower variance ≈ smoother upsampling output.")


## 11  ResNet: Residual Connections and Gradient Flow

With skip connections:
$$
\mathbf{y} = \mathcal{F}(\mathbf{x}, \{W_i\}) + \mathbf{x}
$$

Gradient via chain rule:
$$
\frac{\partial \mathcal{L}}{\partial \mathbf{x}} =
\frac{\partial \mathcal{L}}{\partial \mathbf{y}}
\left(1 + \frac{\partial \mathcal{F}}{\partial \mathbf{x}}\right)
$$

The **+1 term** (identity path) ensures gradients reach shallow layers even if
$\partial\mathcal{F}/\partial\mathbf{x} \approx 0$.


In [ ]:
def simulate_gradient_flow(depth, use_residual=True, block_grad_scale=0.3):
    """Simulate gradient magnitude at each layer depth."""
    grad = 1.0
    grads = [grad]
    for _ in range(depth):
        if use_residual:
            # grad * (1 + ∂F/∂x) where ∂F/∂x ≈ block_grad_scale
            grad = grad * (1.0 + block_grad_scale)
        else:
            # grad * ∂F/∂x — can vanish
            grad = grad * block_grad_scale
        grads.append(grad)
    return grads

depth = 50
grad_res  = simulate_gradient_flow(depth, use_residual=True,  block_grad_scale=0.3)
grad_plain= simulate_gradient_flow(depth, use_residual=False, block_grad_scale=0.9)
grad_van  = simulate_gradient_flow(depth, use_residual=False, block_grad_scale=0.85)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(grad_res,   'C0-', lw=2, label='ResNet (residual)')
axes[0].plot(grad_plain, 'C1--', lw=2, label='Plain (scale=0.9)')
axes[0].plot(grad_van,   'C2:', lw=2, label='Plain (scale=0.85, vanishing)')
axes[0].set_xlabel('Layer depth'); axes[0].set_ylabel('Gradient magnitude (linear)')
axes[0].set_title('Gradient Magnitude vs Depth')
axes[0].legend(); axes[0].set_xlim(0, depth)

axes[1].semilogy(grad_res,   'C0-', lw=2, label='ResNet (residual)')
axes[1].semilogy(grad_plain, 'C1--', lw=2, label='Plain (scale=0.9)')
axes[1].semilogy(grad_van,   'C2:', lw=2, label='Plain (scale=0.85)')
axes[1].set_xlabel('Layer depth'); axes[1].set_ylabel('Gradient magnitude (log)')
axes[1].set_title('Gradient Magnitude vs Depth (log scale)')
axes[1].legend(); axes[1].set_xlim(0, depth)

plt.suptitle('ResNet Skip Connections Prevent Vanishing Gradients', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Bottleneck block parameter count
def bottleneck_params(C, expand=4):
    """ResNet bottleneck: 1×1 → 3×3 → 1×1."""
    p1 = C * (C//expand) * 1  # 1×1 reduce
    p2 = (C//expand)**2 * 9   # 3×3 conv
    p3 = (C//expand) * C * 1  # 1×1 expand
    return p1 + p2 + p3

def standard_block_params(C):
    """Two 3×3 conv layers."""
    return 2 * C**2 * 9

for C in [64, 128, 256, 512]:
    std = standard_block_params(C)
    bot = bottleneck_params(C)
    print(f"C={C:3d}: standard={std:7,d}  bottleneck={bot:6,d}  savings={1-bot/std:.1%}")


## 12  EfficientNet Compound Scaling

Scale width $w$, depth $d$, resolution $r$ jointly:

$$
w = \alpha^\phi, \quad d = \beta^\phi, \quad r = \gamma^\phi
$$

Subject to: $\alpha\cdot\beta^2\cdot\gamma^2 \approx 2$, $\alpha\geq 1, \beta\geq 1, \gamma\geq 1$.

FLOPs scale approximately as $\propto d\cdot w^2\cdot r^2 = (\alpha\beta^2\gamma^2)^\phi \approx 2^\phi$.


In [ ]:
# EfficientNet coefficients (from the paper)
efficientnet_configs = {
    'B0': {'phi': 1.0, 'alpha': 1.2, 'beta': 1.1, 'gamma': 1.15,
           'params_M': 5.3,  'top1': 77.1},
    'B1': {'phi': 1.1, 'alpha': 1.2, 'beta': 1.1, 'gamma': 1.15,
           'params_M': 7.8,  'top1': 79.1},
    'B2': {'phi': 1.2, 'alpha': 1.2, 'beta': 1.1, 'gamma': 1.15,
           'params_M': 9.2,  'top1': 80.1},
    'B3': {'phi': 1.4, 'alpha': 1.2, 'beta': 1.1, 'gamma': 1.15,
           'params_M': 12.0, 'top1': 81.6},
    'B4': {'phi': 1.6, 'alpha': 1.2, 'beta': 1.1, 'gamma': 1.15,
           'params_M': 19.0, 'top1': 82.9},
    'B5': {'phi': 1.8, 'alpha': 1.2, 'beta': 1.1, 'gamma': 1.15,
           'params_M': 30.0, 'top1': 83.6},
    'B6': {'phi': 2.0, 'alpha': 1.2, 'beta': 1.1, 'gamma': 1.15,
           'params_M': 43.0, 'top1': 84.0},
    'B7': {'phi': 2.2, 'alpha': 1.2, 'beta': 1.1, 'gamma': 1.15,
           'params_M': 66.0, 'top1': 84.3},
}

names = list(efficientnet_configs.keys())
params = [c['params_M'] for c in efficientnet_configs.values()]
top1   = [c['top1']     for c in efficientnet_configs.values()]
phis   = [c['phi']      for c in efficientnet_configs.values()]

# Constraint verification
alpha, beta, gamma = 1.2, 1.1, 1.15
constraint = alpha * beta**2 * gamma**2
print(f"Compound constraint α·β²·γ² = {constraint:.4f} (should ≈ 2.0)")

# Compute theoretical widths/depths/resolutions
print("\nModel  φ     width   depth   resolution")
print("-" * 45)
for name, cfg in efficientnet_configs.items():
    phi = cfg['phi']
    w = alpha**phi; d = beta**phi; r = gamma**phi
    print(f"{name}   {phi:.1f}  {w:.2f}x   {d:.2f}x   {r:.2f}x")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(names, params, color='C0', alpha=0.8)
axes[0].set_xlabel('Model'); axes[0].set_ylabel('Parameters (M)')
axes[0].set_title('EfficientNet Parameter Count by Variant')

axes[1].scatter(params, top1, s=100, c=phis, cmap='viridis', zorder=5)
for name, p, t in zip(names, params, top1):
    axes[1].annotate(name, (p, t), textcoords='offset points', xytext=(5,3), fontsize=9)
axes[1].set_xlabel('Parameters (M)'); axes[1].set_ylabel('Top-1 Accuracy (%)')
axes[1].set_title('Accuracy vs Parameters (colour = φ)')
sm = plt.cm.ScalarMappable(cmap='viridis',
     norm=plt.Normalize(min(phis), max(phis)))
plt.colorbar(sm, ax=axes[1], label='φ (compound coefficient)')
plt.tight_layout()
plt.suptitle('EfficientNet Compound Scaling', fontsize=13, fontweight='bold', y=1.01)
plt.show()


## 13  ViT Patch Embedding ≡ Strided Convolution

A Vision Transformer's patch embedding (linear projection of $P\times P$ patches)
is **mathematically equivalent** to:

```python
nn.Conv2d(C_in, d_model, kernel_size=P, stride=P)
```

This equivalence is not just an implementation trick — it shows CNNs and ViTs share
a common first operation.  The key inductive bias difference is what comes *after*.


In [ ]:
def patch_embed_linear(X, P, d_model):
    """
    X: (C, H, W) image
    P: patch size
    Returns: (num_patches, d_model) token sequence
    """
    C, H, W = X.shape
    assert H % P == 0 and W % P == 0
    nH, nW = H // P, W // P
    # Extract patches: (nH*nW, C*P*P)
    patches = []
    for i in range(nH):
        for j in range(nW):
            patch = X[:, i*P:(i+1)*P, j*P:(j+1)*P].ravel()
            patches.append(patch)
    patches = np.stack(patches)  # (N_patches, C*P*P)
    # Random projection matrix (the "linear" in patch embedding)
    proj = np.random.randn(C*P*P, d_model) / np.sqrt(C*P*P)
    return patches @ proj

def patch_embed_conv(X, P, d_model, W_conv):
    """
    Equivalent to Conv2d(C, d_model, kernel_size=P, stride=P).
    W_conv: (d_model, C, P, P)
    Returns: (num_patches, d_model)
    """
    C, H, W = X.shape
    nH, nW = H // P, W // P
    out = np.zeros((nH * nW, d_model))
    idx = 0
    for i in range(nH):
        for j in range(nW):
            patch = X[:, i*P:(i+1)*P, j*P:(j+1)*P]  # (C, P, P)
            for d in range(d_model):
                out[idx, d] = np.sum(patch * W_conv[d])
            idx += 1
    return out

# Verify equivalence
C, H, W, P, d_model = 3, 16, 16, 4, 8
np.random.seed(99)
X_img = np.random.randn(C, H, W)

# Shared random weights: proj matrix = conv kernel reshaped
proj = np.random.randn(C*P*P, d_model) / np.sqrt(C*P*P)
W_conv = proj.T.reshape(d_model, C, P, P)

# Linear path
patches_flat = []
for i in range(H//P):
    for j in range(W//P):
        patches_flat.append(X_img[:, i*P:(i+1)*P, j*P:(j+1)*P].ravel())
patches_flat = np.stack(patches_flat)
out_linear = patches_flat @ proj

# Conv path
out_conv = patch_embed_conv(X_img, P, d_model, W_conv)

print(f"Patch embedding output shape: {out_linear.shape}")
print(f"Conv embedding output shape:  {out_conv.shape}")
print(f"Max absolute difference:      {np.max(np.abs(out_linear - out_conv)):.2e}")
print("Are they equivalent?", np.allclose(out_linear, out_conv, atol=1e-12))

# Summary table
print("\n" + "─"*60)
print("Inductive bias comparison: CNN vs ViT")
print("─"*60)
print(f"{'Property':<28} {'CNN':<16} {'ViT'}")
print("─"*60)
rows = [
    ("Translation equiv.", "Built-in",    "Learned (pos embed)"),
    ("Local connectivity", "Yes (kernel)", "No (full attention)"),
    ("Parameter sharing", "Yes (shared K)","No (different heads)"),
    ("Scale to data",      "Good (<1M)",  "Needs >10M samples"),
    ("Long-range deps.",   "Many layers", "Single layer"),
    ("Patch embed layer",  "IS the conv", "= strided Conv2d"),
]
for r in rows:
    print(f"  {r[0]:<26} {r[1]:<16} {r[2]}")


## 14  ConvNeXt: A CNN Matching ViT Performance

[Liu et al. 2022] systematically ablated ViT training recipes applied to ResNet-50.
Each change was evaluated independently:


In [ ]:
# ConvNeXt improvement roadmap from the paper (approximate accuracy gains)
changes = [
    ("ResNet-50 baseline",           78.8),
    ("+ Training recipe (300 ep)",   79.4),
    ("+ Macro design (stage ratio)", 79.5),
    ("+ ResNeXt-ify (group conv)",   79.9),
    ("+ Inverted bottleneck",        80.6),
    ("+ Large kernel (7×7)",         80.6),
    ("+ Micro design (LayerNorm)",   81.3),
    ("+ GELU activation",            81.3),
    ("+ Fewer activations",          81.4),
    ("+ Fewer norms",                81.5),
    ("+ Separate downsampling",      82.0),
    ("= ConvNeXt-T",                 82.1),
]

labels = [c[0] for c in changes]
accs   = [c[1] for c in changes]
gains  = [0.0] + [accs[i] - accs[i-1] for i in range(1, len(accs))]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(range(len(labels)), accs, color='C0', alpha=0.7)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('ImageNet Top-1 Accuracy (%)')
ax.set_title('ConvNeXt: Step-by-Step Modernisation from ResNet to SOTA CNN')
ax.set_xlim(78, 82.5)

# Annotate gains
for i, (bar, gain) in enumerate(zip(bars, gains)):
    x = bar.get_width()
    color = 'C2' if gain > 0 else 'gray'
    label = f'+{gain:.1f}' if gain > 0 else f'{gain:.1f}' if gain < 0 else ''
    if label:
        ax.text(x + 0.02, i, label, va='center', fontsize=8, color=color, fontweight='bold')

ax.axvline(accs[-1], color='C3', linestyle='--', lw=1.5, label=f'ConvNeXt: {accs[-1]}%')
ax.axvline(accs[0], color='gray', linestyle=':', lw=1.5, label=f'ResNet baseline: {accs[0]}%')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Total improvement: +{accs[-1]-accs[0]:.1f}% accuracy")
print(f"Key insight: standard CNNs can match ViTs with identical training recipes")
print(f"Key insight: 7×7 depthwise conv in each block mimics ViT's attention span")


## Summary

| Concept | Key Equation | Computational Note |
|---|---|---|
| Cross-correlation | $(X\star K)[i,j]=\sum_{m,n}X[i+m,j+n]K[m,n]$ | What DL frameworks compute |
| Toeplitz repr. | $\mathbf{y}=\mathbf{T}\mathbf{x}$ | Rank $\leq k$, sparse |
| FFT conv | $\mathcal{F}\{f*g\}=\mathcal{F}\{f\}\cdot\mathcal{F}\{g\}$ | $O(N\log N)$ vs $O(Nk^2)$ |
| Backprop dX | transposed conv with $K^{\text{flip}}$ | same op, different kernel |
| Backprop dK | cross-corr$(X, dY)$ | accumulates over batch |
| im2col | $Y = WP$ (GEMM) | memory ↑ but GPU fast |
| BatchNorm bwd | analytical 3-term formula | avoid double-backward bugs |
| DW-Sep reduction | $1/C_{\text{out}} + 1/k^2$ | ~8–9x for k=3, large $C$ |
| Dilated RF | $d(k-1)+1$ | exponential via stacking |
| Trans. conv | adjoint of conv | checkerboard: $k\%s=0$ |
| ResNet bwd | $1 + \partial\mathcal{F}/\partial x$ | +1 prevents vanishing |
| EfficientNet | $w=\alpha^\phi, d=\beta^\phi, r=\gamma^\phi$ | FLOP-efficient scaling |
| ViT patch embed | $=$ Conv2d(C, d, k=P, s=P) | same math, different bias |

**Next:** `exercises.ipynb` for hands-on implementations and proofs.
